# Decision Tree Regression for Abnormal Returns Prediction
## Constrained Tree (max_leaf_nodes=50, min_samples_split=100) - Two Features with Monthly Training

This notebook implements a constrained decision tree regression model (max_leaf_nodes=50, min_samples_split=100) to predict abnormal returns using net sentiment and log volume features.
The model is trained once per month (at month-end) and used to predict all days in the following month.

In [ ]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [ ]:
data = pd.read_pickle(INPUT_DATA)

In [ ]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES + ['date', 'ticker']].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

# In-Sample Decision Tree Regression (Constrained)

In [ ]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit constrained decision tree regression model (in-sample)
dt_model = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
dt_model.fit(X, y)

# Make predictions
y_pred = dt_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Constrained Decision Tree Regression Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"\nTree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")
print(f"\nFeature importances:")
for feature, importance in zip(FEATURES, dt_model.feature_importances_):
    print(f"  {feature}: {importance:.6f}")

# Out-of-Sample Predictions

In [ ]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year (in days) for rolling window
WINDOW_21 = 21    # One trading month (in days) for rolling window

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Create a year-month column for grouping
model_data['year_month'] = model_data['date'].dt.to_period('M')

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")

In [ ]:
# Initialize storage for predictions
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []

# Loop through OOS months (train once per month)
for month_idx, pred_month in enumerate(oos_months):
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]

    if len(month_dates) == 0:
        continue

    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)

    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        continue
    last_train_date = train_dates[-1]

    # 1. EXPANDING WINDOW: Train on all data up to end of previous month
    train_mask_exp = model_data['date'] <= last_train_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]

    if len(X_train_exp) > 0:
        dt_exp = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
        dt_exp.fit(X_train_exp, y_train_exp)

        # Use this model to predict for all days in the month
        for pred_date in month_dates:
            test_mask = model_data['date'] == pred_date
            X_test = model_data.loc[test_mask, FEATURES]

            if len(X_test) == 0:
                continue

            test_indices = model_data.index[test_mask]
            y_pred_exp = dt_exp.predict(X_test)

            for idx, pred in zip(test_indices, y_pred_exp):
                predictions_expanding.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_expanding': pred
                })

    # 2. ROLLING 252-DAY WINDOW: Train on last 252 trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx >= WINDOW_252:
        start_date_252 = unique_dates[last_train_date_idx - WINDOW_252 + 1]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] <= last_train_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]

        if len(X_train_252) > 0:
            dt_252 = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
            dt_252.fit(X_train_252, y_train_252)

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                test_indices = model_data.index[test_mask]
                y_pred_252 = dt_252.predict(X_test)

                for idx, pred in zip(test_indices, y_pred_252):
                    predictions_rolling_252.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_252': pred
                    })

    # 3. ROLLING 21-DAY WINDOW: Train on last 21 trading days before the month
    if last_train_date_idx >= WINDOW_21:
        start_date_21 = unique_dates[last_train_date_idx - WINDOW_21 + 1]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] <= last_train_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]

        if len(X_train_21) > 0:
            dt_21 = DecisionTreeRegressor(max_leaf_nodes=50, min_samples_split=100, random_state=42)
            dt_21.fit(X_train_21, y_train_21)

            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]

                if len(X_test) == 0:
                    continue

                test_indices = model_data.index[test_mask]
                y_pred_21 = dt_21.predict(X_test)

                for idx, pred in zip(test_indices, y_pred_21):
                    predictions_rolling_21.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_21': pred
                    })

    # Progress update
    if (month_idx + 1) % 12 == 0:
        print(f"Processed {month_idx + 1}/{len(oos_months)} months ({100 * (month_idx + 1) / len(oos_months):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

In [ ]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

In [ ]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_constrained_monthly.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
if os.path.exists(OUTPUT_FILE):
    print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")